# 05 — LLM as Judge

## Why this notebook exists

In **notebook 04** we built a regression gate using only deterministic graders — exact match, `contains`, structured validation, golden outputs. Those graders are fast, free, and fully reproducible. But they break down the moment the task produces *open-ended text*: two summaries can be equally faithful and concise while sharing almost no words. An `exact_match` grader marks one correct and one wrong based purely on whether the string matches a reference. A `contains` grader tells us a keyword appears, not whether the output is actually good.

LLM-as-judge plugs that gap: we ask a language model to score an output against an explicit rubric, returning a structured score and a rationale we can read. This notebook introduces the technique, shows you how to wire it into the harness from notebook 03 as just another grader, and — critically — shows you where it goes wrong and how to check whether your judge can be trusted.

This is the **first notebook in the series that requires an OpenAI API key.** An early guard cell will stop and print instructions if the key is missing.

## What you'll learn

- Why deterministic graders mis-score open-ended outputs, motivating the need for a judge.
- How to set `OPENAI_API_KEY` and what the guard cell does when it is absent.
- How to define `make_llm_judge(rubric, client)` — a factory that returns a grader with the same `(example, output) -> Score` signature as every other grader in this series.
- How to write a concrete rubric, run the judge on good and bad outputs, and read the `Score` with its `rationale` field.
- How to plug `make_llm_judge` into `run_eval` alongside deterministic graders.
- How to define `judge_pairwise(client, prompt, output_a, output_b)` and why relative judgments are often more reliable than absolute scores.
- The three main traps: **position bias**, **verbosity bias**, and **self-preference / non-determinism** — with a concrete demonstration of position bias.
- How to validate the judge against a small human-labeled set and compute an agreement rate, so the judge is not just another untested component.

## 1. Setup + API Key Guard

This notebook uses `openai` for the LLM judge. Install it if needed:

```bash
pip install openai
```

The cells below (a) import everything and re-declare the shared harness primitives inline, then (b) check for `OPENAI_API_KEY`. **If the key is missing, the guard cell prints setup instructions and raises `SystemExit` — all subsequent API-calling cells are safe to skip.**

To get an API key: visit https://platform.openai.com/api-keys, create a key, and export it in your shell before launching Jupyter:

```bash
export OPENAI_API_KEY="sk-..."
```

Anthropic users: you can swap in `anthropic` SDK calls with the same pattern — the `make_llm_judge` factory accepts any callable you pass as `client`. The default model shown here is `"gpt-4o-mini"` (inexpensive and good enough for grading).

In [ ]:
# ── stdlib ────────────────────────────────────────────────────────────────────
import os
import json
from dataclasses import dataclass, field
from typing import Any, Callable

# ── Load OPENAI_API_KEY from a .env file if present; real env vars still win.
from dotenv import load_dotenv
load_dotenv()

# ── openai SDK ────────────────────────────────────────────────────────────────
from openai import OpenAI  # pip install openai

# ══════════════════════════════════════════════════════════════════════════════
# Shared harness — re-declared inline so this notebook is self-contained.
# These match the canonical definitions from notebooks 03 & 04 exactly.
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Score:
    key: str
    score: float          # normalised to [0, 1]
    passed: bool
    comment: str = ""


@dataclass
class Example:
    input: Any
    expected: Any = None
    metadata: dict = field(default_factory=dict)


@dataclass
class ExampleResult:
    example: Example
    output: Any
    scores: list[Score] = field(default_factory=list)


@dataclass
class EvalReport:
    results: list[ExampleResult]

    @property
    def pass_rate(self) -> float:
        all_scores = [s for r in self.results for s in r.scores]
        if not all_scores:
            return 0.0
        return sum(s.passed for s in all_scores) / len(all_scores)

    @property
    def mean_score(self) -> float:
        all_scores = [s for r in self.results for s in r.scores]
        if not all_scores:
            return 0.0
        return sum(s.score for s in all_scores) / len(all_scores)

    def summary_table(self) -> None:
        """Print a simple per-example summary table."""
        print(f"{'#':<4} {'passed':<8} {'mean score':<12} {'comment'}")
        print("-" * 60)
        for i, r in enumerate(self.results):
            scores = r.scores
            if not scores:
                print(f"{i:<4} {'—':<8} {'—':<12} no graders")
                continue
            passed = all(s.passed for s in scores)
            mean = sum(s.score for s in scores) / len(scores)
            comments = "; ".join(s.comment for s in scores if s.comment)
            print(f"{i:<4} {'✓' if passed else '✗':<8} {mean:<12.3f} {comments[:60]}")
        print("-" * 60)
        print(f"Pass rate: {self.pass_rate:.1%}   Mean score: {self.mean_score:.3f}")


Grader = Callable[[Example, Any], Score]


def run_eval(
    agent: Callable[[Any], Any],
    dataset: list[Example],
    graders: list[Grader],
) -> EvalReport:
    """Run `agent` on every `Example`, apply every grader, return a report."""
    results: list[ExampleResult] = []
    for example in dataset:
        output = agent(example.input)
        scores = [grader(example, output) for grader in graders]
        results.append(ExampleResult(example=example, output=output, scores=scores))
    return EvalReport(results=results)


print("Harness re-declared OK.")

In [ ]:
# ── API key guard ─────────────────────────────────────────────────────────────
# This cell must run before any cell that calls the OpenAI API.
# If OPENAI_API_KEY is not set, it prints setup instructions and stops the
# notebook so subsequent cells that call the API are safe to skip.

_api_key = os.getenv("OPENAI_API_KEY")

if not _api_key:
    print(
        "┌─────────────────────────────────────────────────────────────────┐\n"
        "│  OPENAI_API_KEY is not set.                                     │\n"
        "│                                                                 │\n"
        "│  This is the first notebook in the series that needs a key.    │\n"
        "│  Steps:                                                         │\n"
        "│    1. Visit https://platform.openai.com/api-keys               │\n"
        "│    2. Create a new secret key.                                  │\n"
        "│    3. In your terminal (before launching Jupyter):              │\n"
        "│         export OPENAI_API_KEY=\"sk-...\"                         │\n"
        "│    4. Restart the Jupyter kernel and re-run from the top.       │\n"
        "│                                                                 │\n"
        "│  Anthropic alternative: replace `from openai import OpenAI`    │\n"
        "│  with `import anthropic` and adapt the client calls in         │\n"
        "│  make_llm_judge / judge_pairwise to use the Messages API.       │\n"
        "└─────────────────────────────────────────────────────────────────┘"
    )
    raise SystemExit(
        "Set OPENAI_API_KEY and restart the kernel to continue."
    )

client = OpenAI()  # reads OPENAI_API_KEY from environment automatically
DEFAULT_MODEL = "gpt-4o-mini"

print(f"OpenAI client ready. Default model: {DEFAULT_MODEL}")
print("Key found (first 8 chars):", _api_key[:8] + "…")

## 2. Why Deterministic Graders Fall Short

Suppose our agent summarises a paragraph. We have a reference summary. Let's define two *clearly good* candidate summaries — both faithful, concise, and complete — and watch `exact_match` and `contains` score them.

In [ ]:
# ── Source paragraph and reference ────────────────────────────────────────────
SOURCE = (
    "The Apollo 11 mission, launched on July 16, 1969, successfully landed "
    "astronauts Neil Armstrong and Buzz Aldrin on the Moon on July 20. "
    "Armstrong became the first human to walk on the lunar surface, followed "
    "by Aldrin. Michael Collins orbited the Moon in the command module while "
    "his crewmates explored the surface. The crew returned safely to Earth "
    "on July 24, 1969."
)

REFERENCE = (
    "Apollo 11 (July 1969) landed Armstrong and Aldrin on the Moon; "
    "Collins orbited above. Armstrong was first to walk on the surface. "
    "The crew returned safely on July 24."
)

# Two summaries — both clearly good, written differently
SUMMARY_A = (
    "In July 1969, Apollo 11 brought Neil Armstrong and Buzz Aldrin to the "
    "lunar surface. Armstrong stepped out first. Michael Collins remained in "
    "orbit. All three astronauts returned to Earth safely on July 24th."
)

SUMMARY_B = (
    "Apollo 11 successfully landed on the Moon on July 20, 1969. "
    "Armstrong and Aldrin walked on the surface while Collins orbited. "
    "The mission concluded with a safe splashdown on July 24, 1969."
)

print("Source, reference, and two good candidate summaries defined.")

In [ ]:
# ── Deterministic graders ─────────────────────────────────────────────────────

def exact_match(example: Example, output: str) -> Score:
    passed = output.strip() == str(example.expected).strip()
    return Score(key="exact_match", score=1.0 if passed else 0.0, passed=passed)


def contains_keywords(keywords: list[str]) -> Grader:
    """Grader factory: passes if ALL keywords appear in the output."""
    def grader(example: Example, output: str) -> Score:
        found = [kw for kw in keywords if kw.lower() in output.lower()]
        score = len(found) / len(keywords)
        passed = score == 1.0
        comment = f"found {len(found)}/{len(keywords)}: {found}"
        return Score(key="contains_keywords", score=score, passed=passed, comment=comment)
    return grader


# Evaluate both summaries with the reference as `expected`
example_ref = Example(input=SOURCE, expected=REFERENCE)

kw_grader = contains_keywords(["Armstrong", "Aldrin", "Collins", "July 24"])

print("=== Summary A ===")
em_a = exact_match(example_ref, SUMMARY_A)
kw_a = kw_grader(example_ref, SUMMARY_A)
print(f"  exact_match  → passed={em_a.passed}, score={em_a.score}")
print(f"  contains_kw  → passed={kw_a.passed}, score={kw_a.score:.2f}, {kw_a.comment}")

print()
print("=== Summary B ===")
em_b = exact_match(example_ref, SUMMARY_B)
kw_b = kw_grader(example_ref, SUMMARY_B)
print(f"  exact_match  → passed={em_b.passed}, score={em_b.score}")
print(f"  contains_kw  → passed={kw_b.passed}, score={kw_b.score:.2f}, {kw_b.comment}")

print()
print(
    "Both summaries are accurate and well-written.\n"
    "exact_match scores both 0 (neither matches the reference string exactly).\n"
    "contains_keywords scores both 1.0 — but can't tell us HOW good they are.\n"
    "We need a grader that reads the text and reasons about quality."
)

## 3. Rubric Grading — `make_llm_judge`

A rubric judge takes an explicit scoring criterion written in prose, sends the output (and the original input + expected reference) to a language model, and asks for a structured JSON response: `{"score": <0–1 float>, "passed": <bool>, "rationale": <string>}`. The rationale is stored in `Score.comment` so we can read *why* the judge gave each score.

`make_llm_judge` is a **factory**: it takes the rubric and configuration, and returns a grader function with the standard `(example, output) -> Score` signature — making it a drop-in for `run_eval` alongside any deterministic grader.

### Try it

In [ ]:
def make_llm_judge(
    rubric: str,
    client: OpenAI,
    model: str = "gpt-4o-mini",
    key: str = "llm_judge",
) -> Grader:
    """Return a grader that scores an output against `rubric` using an LLM.

    The returned grader has signature (example, output) -> Score and can be
    passed directly to run_eval alongside deterministic graders.

    The LLM is prompted with:
      - The rubric
      - example.input (the prompt given to the agent)
      - example.expected (the reference output, if any)
      - The candidate output to score

    It must respond with JSON: {"score": <0-1 float>, "passed": <bool>,
    "rationale": <string>}. score is normalised to [0, 1] before returning.
    """

    system_prompt = (
        "You are a precise, impartial evaluator. "
        "Score the candidate output against the rubric. "
        "Respond with ONLY valid JSON matching this schema:\n"
        '{"score": <float 0-1>, "passed": <bool>, "rationale": <string>}\n'
        "No markdown fences, no extra keys."
    )

    def grader(example: Example, output: Any) -> Score:
        user_message = (
            f"## Rubric\n{rubric}\n\n"
            f"## Input given to the agent\n{example.input}\n\n"
        )
        if example.expected is not None:
            user_message += f"## Reference output\n{example.expected}\n\n"
        user_message += f"## Candidate output to score\n{output}\n\n"
        user_message += "Respond with JSON only."

        response = client.chat.completions.create(
            model=model,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
            ],
            temperature=0,
        )

        raw = response.choices[0].message.content
        parsed = json.loads(raw)

        raw_score = float(parsed["score"])
        normalised = max(0.0, min(1.0, raw_score))  # clamp to [0, 1]
        passed = bool(parsed["passed"])
        rationale = str(parsed.get("rationale", ""))

        return Score(key=key, score=normalised, passed=passed, comment=rationale)

    return grader


print("make_llm_judge defined.")

In [ ]:
# ── Rubric for summary quality ─────────────────────────────────────────────────
SUMMARY_RUBRIC = (
    "Score the summary on two equally-weighted dimensions (each 0–0.5, "
    "combined into a single 0–1 score):\n"
    "  1. FAITHFULNESS: Every claim in the summary must be supported by the "
    "source text. Penalise hallucinated facts, wrong dates, or wrong names.\n"
    "  2. CONCISENESS: The summary should convey the essential information "
    "without padding or irrelevant detail.\n"
    "A score >= 0.7 should be marked passed=true. "
    "Score < 0.7 should be marked passed=false."
)

judge = make_llm_judge(rubric=SUMMARY_RUBRIC, client=client)

# A clearly bad summary: hallucinated detail (wrong astronaut count, wrong date)
BAD_SUMMARY = (
    "The Apollo 11 mission in August 1969 sent four astronauts to the Moon. "
    "Armstrong was the only one to walk on the surface. The crew never returned."
)

print("Scoring Summary A (good)…")
score_a = judge(Example(input=SOURCE, expected=REFERENCE), SUMMARY_A)
print(f"  key={score_a.key!r}, score={score_a.score:.3f}, passed={score_a.passed}")
print(f"  rationale: {score_a.comment[:200]}")

print()
print("Scoring Summary B (good)…")
score_b = judge(Example(input=SOURCE, expected=REFERENCE), SUMMARY_B)
print(f"  key={score_b.key!r}, score={score_b.score:.3f}, passed={score_b.passed}")
print(f"  rationale: {score_b.comment[:200]}")

print()
print("Scoring Bad Summary (hallucinated facts)…")
score_bad = judge(Example(input=SOURCE, expected=REFERENCE), BAD_SUMMARY)
print(f"  key={score_bad.key!r}, score={score_bad.score:.3f}, passed={score_bad.passed}")
print(f"  rationale: {score_bad.comment[:200]}")

In [ ]:
# ── Plug the judge into run_eval alongside a deterministic grader ──────────────

dataset = [
    Example(input=SOURCE, expected=REFERENCE),
    Example(input=SOURCE, expected=REFERENCE),
    Example(input=SOURCE, expected=REFERENCE),
]

def summary_agent_good(inp: str) -> str:
    """Stub: always returns Summary A."""
    return SUMMARY_A

def summary_agent_bad(inp: str) -> str:
    """Stub: always returns the hallucinated bad summary."""
    return BAD_SUMMARY

graders = [
    kw_grader,    # deterministic: from section 2
    judge,        # LLM: from this section
]

print("=== Good agent ===")
report_good = run_eval(summary_agent_good, dataset, graders)
report_good.summary_table()

print()
print("=== Bad agent ===")
report_bad = run_eval(summary_agent_bad, dataset, graders)
report_bad.summary_table()

## 4. Pairwise Comparison — `judge_pairwise`

Absolute rubric scores are useful, but they have a calibration problem: two different rubrics, or even two different prompts for the same rubric, can assign very different absolute values to the same output. Pairwise comparison sidesteps this: we show the judge two candidate outputs and ask only *"which is better?"* — a relative judgment that is often more reliable and robust to prompt wording.

`judge_pairwise` returns `"A"`, `"B"`, or `"tie"`.

### Try it

In [ ]:
def judge_pairwise(
    client: OpenAI,
    prompt: str,
    output_a: str,
    output_b: str,
    model: str = "gpt-4o-mini",
) -> str:
    """Ask the LLM which of two outputs is better for `prompt`.

    Returns "A", "B", or "tie".

    `prompt` is the task description / input; output_a and output_b are the
    two candidates. The judge is shown both and must choose.
    """
    system_prompt = (
        "You are a fair evaluator. Given a task prompt and two candidate "
        "outputs (A and B), decide which is better.\n"
        "Respond with ONLY valid JSON: "
        '{"winner": "A" | "B" | "tie", "reason": "<one sentence>"}\n'
        "No markdown fences, no extra keys. "
        "Be objective; do not favour longer responses."
    )
    user_message = (
        f"## Task prompt\n{prompt}\n\n"
        f"## Output A\n{output_a}\n\n"
        f"## Output B\n{output_b}\n\n"
        "Which output better addresses the task? Respond with JSON only."
    )

    response = client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        temperature=0,
    )

    raw = response.choices[0].message.content
    parsed = json.loads(raw)
    winner = str(parsed.get("winner", "tie")).strip().upper()
    if winner not in {"A", "B", "TIE"}:
        winner = "TIE"
    reason = str(parsed.get("reason", ""))
    print(f"  winner={winner!r}, reason: {reason[:120]}")
    return "tie" if winner == "TIE" else winner


print("judge_pairwise defined.")

In [ ]:
# ── Compare Summary A vs Bad Summary ──────────────────────────────────────────
TASK_PROMPT = f"Summarise the following paragraph concisely and faithfully:\n\n{SOURCE}"

print("Comparing good Summary A vs hallucinated Bad Summary:")
winner = judge_pairwise(client, TASK_PROMPT, SUMMARY_A, BAD_SUMMARY)
print(f"Result: {winner!r}  (expected: 'A')\n")

print("Comparing bad Summary vs good Summary A (reversed):")
winner_rev = judge_pairwise(client, TASK_PROMPT, BAD_SUMMARY, SUMMARY_A)
print(f"Result: {winner_rev!r}  (expected: 'B')\n")

print("Comparing two good summaries (A vs B):")
winner_ab = judge_pairwise(client, TASK_PROMPT, SUMMARY_A, SUMMARY_B)
print(f"Result: {winner_ab!r}  (may be 'A', 'B', or 'tie' — both are good)")